In [1]:
import os
import rqdatac
import pandas as pd
import datetime as dt
rqdatac.init(username="license", password="gCKbHurs4dlMyehGC3GVBEYgFsPRZZiVNUWfCJCS9ifEdXYWnBqgopXvtwMg3GdeJxvb02yljxgaEYxhu1pREMs6k4oFmIU5e0Lf4k56THXNJdgY9i90ehi9i_Hh9sDDSHYg3WgNslsvOwIo4Ku66nV2P1T69RprXP0OIqsep3M=F1112RCtTHbSGqqSJUDAyNXbGm-ik0mkYJGwcAKsg8YNX6oj6u_dAnCo2tUYJ6jp7PAtYxCA3p3SXDA5xa4f_X-eZA5T2vbtFqWkHU5QEz6gDnIsCHX5JSkzUIPqToU8rLOD8D3q-MAJICrCnZ8B4y3Hp6X6KCSR_8X8vMddDkc=")
# rqdatac.init(13601611030,'PB123456789')

E:\anaconda\Lib\site-packages\rqdatac\client.py:263: UserWarning: Your account will be expired after  36 days. Please call us at 0755-22676337 to upgrade or purchase or renew your contract.
  warnings.warn("Your account will be expired after  {} days. "


1.期货CTA相关计算过程

In [2]:
# 获取期货收盘价和结算价
def get_fut_price_info(date):
    fut_mkt_df = rqdatac.all_instruments(type='Future', market='cn', date=date)
    fut_mkt_df = fut_mkt_df[(fut_mkt_df['order_book_id'].str.startswith(('IH', 'IF', 'IC', 'IM', 'T', 'A', 'M'))) & ((fut_mkt_df['exchange'] == 'CFFEX') | (fut_mkt_df['exchange'] == 'DCE')) & ((
        fut_mkt_df['symbol'].apply(lambda x: '连续' not in x)) | (fut_mkt_df['symbol'].apply(lambda x: '国债' in x)))].reset_index(drop=True)
    fut_mkt_df = fut_mkt_df[['order_book_id', 'exchange', 'contract_multiplier']].rename(columns={'order_book_id': 'code'})
    fut_mkt_codes = fut_mkt_df['code'].tolist()
    try:
        fut_mkt = rqdatac.get_price(fut_mkt_codes, start_date=date, end_date=date, frequency='1d',
                                    fields=None, adjust_type='pre', skip_suspended=False, market='cn', expect_df=True, time_slice=None).reset_index()
        fut_mkt = fut_mkt[['order_book_id', 'close', 'settlement', 'prev_close', 'prev_settlement']].rename(columns={'order_book_id': 'code', 'prev_close': 'preclose', 'prev_settlement': 'presettlement'})
        fut_mkt_df = pd.merge(fut_mkt_df, fut_mkt, on='code', how='left')
    except:
        pre_date = rqdatac.get_previous_trading_date(date)
        fut_mkt = rqdatac.get_price(fut_mkt_codes, start_date=pre_date, end_date=pre_date, frequency='1d',
                                fields=None, adjust_type='pre', skip_suspended=False, market='cn', expect_df=True, time_slice=None).reset_index()
        fut_mkt = fut_mkt[['order_book_id', 'close', 'settlement']].rename(
            columns={'order_book_id': 'code', 'close': 'preclose', 'settlement': 'presettlement'})
        fut_mkt_df = pd.merge(fut_mkt_df, fut_mkt, on='code', how='left')
        data = pd.read_csv(f'/home/zhanggh/testscripts/test_data/{date}_data.csv', index_col=0).rename(columns={'order_book_id': 'code'})
        # 按照所有列名去除重复值，保留最后一行重复值，且重置索引
        data = data.drop_duplicates(subset=None, keep='last', ignore_index=True)
        data['datetime'] = data['datetime'].astype(str)
        data[['date', 'time']] = data['datetime'].str.split(' ', expand=True)
        data['date'] = data['date'].apply(lambda x: x.replace('-', ''))
        data['time'] = data['time'].apply(lambda x: x.replace(':', ''))
        close_data = data[data['time'] == '150000'].reset_index()
        fut_mkt_df = pd.merge(fut_mkt_df, close_data[['code', 'close']], on='code', how='left')
        data = data[(data['time'] > '140000') & (data['time'] <= '150000')].reset_index()
        grouped = data[['code', 'volume', 'total_turnover']].groupby('code').sum().reset_index()
        grouped['settlement'] = grouped['total_turnover'] / grouped['volume']
        fut_mkt_df = pd.merge(fut_mkt_df, grouped[['code', 'settlement']], on='code', how='left')
        fut_mkt_df['settlement'] = fut_mkt_df['settlement'] / fut_mkt_df['contract_multiplier']
    # print(fut_mkt_df)
    # print(fut_mkt_df[fut_mkt_df['code'] == 'A2605'])
    return fut_mkt_df

# 处理期货交易记录，量化交易收益，cta交易收益
def deal_fut_order(date, product):
    fut_price_info = get_fut_price_info(date)
    order_paths = rf'E:\code\generate_split_system\data\standarddata\fut\{date}\{date}_{product}_fut_order.csv'
    quant_order_pnl = 0
    quant_used_fee = 0
    cta_order_pnl = 0
    cta_used_fee = 0
    if os.path.exists(order_paths):
        data = pd.read_csv(order_paths, index_col=0)
        # if date == '20260311':
        #     print(data)
        if data.shape[0] != 0:
            data = pd.merge(data, fut_price_info, on='code', how='left')
            quant_data = data[data['user'] == 'quant'].reset_index(drop=True)
            if quant_data.shape[0] != 0:
                quant_data['order_pnl'] = quant_data['Direction'] * quant_data['filled_vol'] *  quant_data['contract_multiplier'] * (quant_data['presettlement'] - quant_data['filled_price'])
                # quant_data['order_pnl'] = quant_data['Direction'] * quant_data['filled_vol'] *  quant_data['contract_multiplier'] * (quant_data['preclose'] - quant_data['filled_price'])
                quant_used_fee = quant_data['UsedFee'].sum()
                quant_order_pnl = quant_data['order_pnl'].sum()

            cta_data = data[data['user'] == 'cta'].reset_index(drop=True)
            if cta_data.shape[0] != 0:
                # print(cta_data)
                cta_data['order_pnl'] = cta_data['Direction'] * cta_data['filled_vol'] *  cta_data['contract_multiplier'] * (cta_data['presettlement'] - cta_data['filled_price'])
                # cta_data['order_pnl'] = cta_data['Direction'] * cta_data['filled_vol'] *  cta_data['contract_multiplier'] * (cta_data['preclose'] - cta_data['filled_price'])
                cta_used_fee = cta_data['UsedFee'].sum()
                cta_order_pnl = cta_data['order_pnl'].sum()
                # if date == '20260210':
                #     cta_order_pnl = cta_order_pnl + 3600 if product == 'PBHSZX1H' else cta_order_pnl if product == 'PBPFZX1H' else cta_order_pnl + 7200
    return round(quant_order_pnl, 2), round(quant_used_fee, 2), round(cta_order_pnl, 2), round(cta_used_fee, 2)

# 处理期货持仓信息
def deal_fut_pos(date, product):
    fut_price_info = get_fut_price_info(date)
    pos_paths = rf'E:\code\generate_split_system\data\standarddata\fut\{date}\{date}_{product}_fut_pos.csv'
    quant_hold_value = 0
    quant_hold_pnl = 0
    quant_diff_pnl = 0
    cta_hold_value = 0
    cta_hold_pnl = 0
    cta_diff_pnl = 0
    if os.path.exists(pos_paths) and (os.path.getsize(pos_paths) != 0):
        data = pd.read_csv(pos_paths, index_col=0)
        if data.shape[0] != 0:
            # if date == '20260311':
            #     print(data)
            data = pd.merge(data, fut_price_info, on='code', how='left')
            quant_data = data[data['user'] == 'quant'].reset_index(drop=True)
            if quant_data.shape[0] != 0:
                quant_data['hold_value'] = quant_data['Direction'] * quant_data['vol'] * quant_data['contract_multiplier'] * quant_data['settlement']
                quant_hold_value = quant_data['hold_value'].sum().round(0)
                quant_data['hold_pnl'] = (quant_data['settlement'] - quant_data['presettlement']) * quant_data['vol'] * quant_data['contract_multiplier'] * quant_data['Direction']
                quant_hold_pnl = quant_data['hold_pnl'].sum().round(0)
                quant_data['diff_pnl'] = (quant_data['settlement'] - quant_data['close']) * quant_data['Direction'] * quant_data['vol'] * quant_data['contract_multiplier']
                quant_diff_pnl = quant_data['diff_pnl'].sum().round(0)
            cta_data = data[data['user'] == 'cta'].reset_index(drop=True)
            if cta_data.shape[0] != 0:
                cta_data['hold_value'] = cta_data['Direction'] * cta_data['vol'] * cta_data['contract_multiplier'] * cta_data['settlement']
                cta_hold_value = cta_data['hold_value'].sum().round(0)
                cta_data['hold_pnl'] = (cta_data['settlement'] - cta_data['presettlement']) * cta_data['vol'] * cta_data['contract_multiplier'] * cta_data['Direction']
                # cta_data['hold_pnl'] = (cta_data['close'] - cta_data['preclose']) * cta_data['vol'] * cta_data['contract_multiplier'] * cta_data['Direction']
                cta_hold_pnl = cta_data['hold_pnl'].sum().round(0)
                cta_data['diff_pnl'] = (cta_data['settlement'] - cta_data['close']) * cta_data['Direction'] * cta_data['vol'] * cta_data['contract_multiplier']
                cta_diff_pnl = cta_data['diff_pnl'].sum().round(0)
    return quant_hold_value, quant_hold_pnl, quant_diff_pnl, cta_hold_value, cta_hold_pnl, cta_diff_pnl


cta_date = '20260408'
cta_date = dt.datetime.now().strftime('%Y%m%d') # # 20250911
rk_dates = rqdatac.get_trading_dates(start_date='20250911', end_date=cta_date)
rk_dates = [str(date).replace('-', '') for date in rk_dates]
# print(rk_dates)
products = ['PBTZ2H', 'PBHSZX1H', 'PBPFZX1H'] # 'PBTZ2H', 'PBHSZX1H', 'PBPFZX1H'
cta_df = pd.DataFrame()
for product in products:
    print(product)
    cta_trade_pnls, cta_trade_fees, cta_hold_pnls = [], [], []
    for date in rk_dates:
        _, _, cta_order_pnl, cta_used_fee = deal_fut_order(date, product)
        _, _, _, _, cta_hold_pnl, _ = deal_fut_pos(date, product)
        cta_trade_pnls.append(cta_order_pnl)
        cta_trade_fees.append(cta_used_fee)
        cta_hold_pnls.append(cta_hold_pnl)
    tmp = pd.DataFrame()
    tmp['日期'] = rk_dates
    tmp['产品'] = product
    tmp['产品名'] = tmp['产品'].apply(lambda x: '配邦恒升中性1号' if x == 'PBHSZX1H' else '配邦鹏飞中性1号' if x == 'PBPFZX1H' else '配邦投资二号')
    # tmp['cta交易收益'] = cta_trade_pnls
    tmp['cta交易收益'] = cta_trade_pnls
    tmp['cta交易费用'] = cta_trade_fees
    tmp['cta持仓收益'] = cta_hold_pnls
    tmp['cta费后收益'] = tmp['cta交易收益'] - tmp['cta交易费用'] + tmp['cta持仓收益']
    tmp['cta累计费后收益'] = tmp['cta费后收益'].cumsum()
    # tot_pnl = round(sum(cta_pnls), 2)
    # print(f'{product} 总收益：{tot_pnl}')
    cta_df = pd.concat([cta_df, tmp])
cta_df = cta_df.sort_values(by='日期').reset_index(drop=True)
cta_df.tail(3)
# df


PBTZ2H
PBHSZX1H
PBPFZX1H


,日期,产品,产品名,cta交易收益,cta交易费用,cta持仓收益,cta费后收益,cta累计费后收益
486,20260521,PBTZ2H,配邦投资二号,405640.0,1336.87,-105280.0,299023.13,-235812.35
487,20260521,PBHSZX1H,配邦恒升中性1号,0.0,0.00,0.0,0.00,-233448.17
488,20260521,PBPFZX1H,配邦鹏飞中性1号,0.0,0.00,0.0,0.00,-336161.14


2.基差收敛程度

In [3]:
# 基差收敛
def analysis(exam_date, benchmark, hengsheng_strategy_paths, hengsheng_pos_paths):
    preday = rqdatac.get_previous_trading_date(exam_date).strftime('%Y%m%d')

    try:
        ind_data = rqdatac.get_price(benchmark, start_date=exam_date, end_date=exam_date, frequency='1d',
                                        fields=['close', 'prev_close'], adjust_type='pre', skip_suspended=False, market='cn', expect_df=True, time_slice=None).reset_index().rename(columns={'order_book_id': 'code'})
    except:
        print('realtime')
        ind_data_today = rqdatac.current_minute(benchmark, fields = ['close'], skip_suspended=False).reset_index().rename(columns={'order_book_id': 'code'})
        ind_data_preday = rqdatac.get_price(benchmark, start_date=preday, end_date=preday, frequency='1d',
                                        fields=['close'], adjust_type='pre', skip_suspended=False, market='cn', expect_df=True, time_slice=None).reset_index().rename(columns={'order_book_id': 'code', 'close': 'prev_close'})
        ind_data = pd.merge(ind_data_today[['code', 'close']], ind_data_preday[['code', 'prev_close']])
    ind_pct =  ind_data.loc[0, 'close'] / ind_data.loc[0, 'prev_close'] - 1

    st_data = pd.read_csv(hengsheng_strategy_paths)
    pos_data = pd.read_csv(hengsheng_pos_paths)
    pos_data['code'] = pos_data['code'].apply(lambda x: str(x).zfill(6))
    pos_data['code'] = pos_data['code'].apply(lambda x: x + '.XSHG' if x[:1] == '6'  else x + '.XSHE')
    data  = pd.merge(st_data, pos_data, how='outer').fillna(0)
    exam_codes = data['code'].unique().tolist()
    try:
        exam_data = rqdatac.get_price(exam_codes, start_date=exam_date, end_date=exam_date, frequency='1d',
                                        fields=['close', 'prev_close'], adjust_type='pre', skip_suspended=False, market='cn', expect_df=True, time_slice=None).reset_index().rename(columns={'order_book_id': 'code'})
    except:
        print('realtime')
        exam_data_today = rqdatac.current_minute(exam_codes, fields = ['close'], skip_suspended=False).reset_index().rename(columns={'order_book_id': 'code'})
        exam_data_preday = rqdatac.get_price(exam_codes, start_date=preday, end_date=preday, frequency='1d',
                                        fields=['close'], adjust_type='pre', skip_suspended=False, market='cn', expect_df=True, time_slice=None).reset_index().rename(columns={'order_book_id': 'code', 'close': 'prev_close'})
        exam_data = pd.merge(exam_data_today[['code', 'close']], exam_data_preday[['code', 'prev_close']])

    data = pd.merge(data, exam_data[['code', 'close', 'prev_close']], how='outer').fillna(0)
    # print(data)
    # data.to_csv('/home/zhanggh/testscripts/get_price.csv')
    # data.to_csv('/home/zhanggh/testscripts/current_minute.csv')
    mean_w = 1 / len(data)
    data['mean_w'] = mean_w
    data['hold_w'] = (data['prev_close'] * data['hold']) / (data['prev_close'] * data['hold']).sum()

    strategy_pnl = (data['w'] * (data['close'] / data['prev_close'] - 1)).sum()
    print((data['hold'] * (data['close'] - data['prev_close'])).sum())
    pos_pnl = (data['hold_w'] * (data['close'] / data['prev_close'] - 1)).sum()
    mean_strategy_pnl = (data['mean_w'] * (data['close'] / data['prev_close'] - 1)).sum()
    def format_float(x):
        return format(x, '.2%')

    data['pnl'] = data['close'] / data['prev_close'] - 1

    data['mean_pnl'] = data['mean_w'] * data['pnl']
    data['hold_pnl'] = data['hold_w'] * data['pnl']
    print(f'日期: {exam_date}')
    print(f'指数收益率为：{format_float(ind_pct)}, 等权分配策略收益率：{format_float(mean_strategy_pnl)}, 策略收益率：{format_float(strategy_pnl)}, 实际持仓收益率：{format_float(pos_pnl)}')
    print(f'等权分配策略超额：{format_float(mean_strategy_pnl-ind_pct)}, 策略超额：{format_float(strategy_pnl-ind_pct)}, 实际持仓超额：{format_float(pos_pnl-ind_pct)}')

    n_1 = data[data['hold_w'] >= mean_w]['hold_pnl'].sum()
    n_2 = data[data['hold_w'] >= mean_w]['mean_pnl'].sum()
    n_3 = data[data['hold_w'] < mean_w]['hold_pnl'].sum()
    n_4 = data[data['hold_w'] < mean_w]['mean_pnl'].sum()
    num = len(data[data['hold_w'] >= mean_w])
    num_pct = num / len(data)
    print(f'等权重为：{format_float(mean_w)}, 实际持仓权重超过等权重的比例为: {format_float(num_pct)}')
    print(f'实际持仓中权重超过等权重部分的收益率为：{format_float(n_1)}, 对应的等权重收益率为：{format_float(n_2)}')
    print(f'实际持仓中权重小于等权重部分的收益率为：{format_float(n_3)}, 对应的等权重收益率为：{format_float(n_4)}')

def cal_ind_fut_data(date, fut_code, ind_code):
    pre_date = rqdatac.get_previous_trading_date(date)
    try:
        df_fut = rqdatac.get_price(fut_code, start_date=date, end_date=date, frequency='1d', fields=None, adjust_type='pre', skip_suspended=False, expect_df=True, time_slice=None, market='cn').reset_index()
        df_ind = rqdatac.get_price(ind_code, start_date=date, end_date=date, frequency='1d', fields=None, adjust_type='pre', skip_suspended=False, expect_df=True, time_slice=None, market='cn').reset_index()

    except Exception as e:
        # print(f"米筐获取期货结算价失败 ({date}): {e}")
        # print('使用记录盘中分钟线实时行情计算结算价')
        df_fut = rqdatac.get_price(fut_code, start_date=pre_date, end_date=pre_date, frequency='1d',
                                    fields=None, adjust_type='pre', skip_suspended=False, market='cn', expect_df=True, time_slice=None).reset_index()

        df_fut['prev_close'] = df_fut['close']
        df_fut['prev_settlement'] = df_fut['settlement']
        data = pd.read_csv(
            rf'E:\code\generate_split_system\data\raw\real_minute\{date}_data.csv', index_col=0).rename(columns={'order_book_id': 'code'})
        # 按照所有列名去除重复值，保留最后一行重复值，且重置索引
        data = data.drop_duplicates(
            subset=None, keep='last', ignore_index=True)
        data['datetime'] = data['datetime'].astype(str)
        data[['date', 'time']] = data['datetime'].str.split(
            ' ', expand=True)
        data['date'] = data['date'].apply(lambda x: x.replace('-', ''))
        data['time'] = data['time'].apply(lambda x: x.replace(':', ''))
        # close_data = data[data['time'] == '150000'].reset_index()
        data = data[(data['time'] > '140000') & (data['time'] <= '150000')].reset_index()
        grouped = data[['code', 'volume', 'total_turnover']].groupby('code').sum().reset_index()
        contract_multiplier = 300 if fut_code.startswith(('IF', 'IH')) else 200 if fut_code.startswith(('IC', 'IM')) else None
        grouped['settlement'] = grouped['total_turnover'] / grouped['volume'] / contract_multiplier
        settlement = grouped[grouped['code'] == fut_code]['settlement'].values[0]
        df_fut['settlement'] = settlement

        df_ind = rqdatac.get_price(ind_code, start_date=pre_date, end_date=pre_date, frequency='1d', fields=None, adjust_type='pre', skip_suspended=False, expect_df=True, time_slice=None, market='cn').reset_index()
        df_ind['prev_close'] = df_ind['close']
        df = rqdatac.current_minute(ind_code, skip_suspended=False).reset_index()
        ind_close = df.loc[0, 'close']
        df_ind['close'] = ind_close
    return df_fut, df_ind

def get_ind_fut_contrat(date, id):
    fut_mkt_df = rqdatac.all_instruments(type='Future', market='cn', date=date)
    # fut_mkt_df = fut_mkt_df[(fut_mkt_df['order_book_id'].str.startswith(('IH', 'IF', 'IC', 'IM'))) & (fut_mkt_df['symbol'].apply(lambda x: '连续' not in x))].reset_index(drop=True)
    fut_mkt_df = fut_mkt_df[(fut_mkt_df['order_book_id'].str.startswith(id)) & (fut_mkt_df['symbol'].apply(lambda x: '连续' not in x))].reset_index(drop=True)
    ind_fut_dic = {}
    for ind_code in fut_mkt_df['underlying_order_book_id'].unique().tolist():
        tmp = fut_mkt_df[fut_mkt_df['underlying_order_book_id'] == ind_code].reset_index(drop=True)
        ind_fut_dic[ind_code] = tmp['order_book_id'].tolist()
    return ind_fut_dic


def cal_basis(date, id):
    # df_fut = rqdatac.get_price(fut_code, start_date=date, end_date=date, frequency='1d', fields=None, adjust_type='pre', skip_suspended=False, expect_df=True, time_slice=None, market='cn').reset_index()
    # df_ind = rqdatac.get_price(ind_code, start_date=date, end_date=date, frequency='1d', fields=None, adjust_type='pre', skip_suspended=False, expect_df=True, time_slice=None, market='cn').reset_index()
    ind_fut_dic = get_ind_fut_contrat(date, id)
    for ind_code in ind_fut_dic:
        for fut_code in ind_fut_dic.get(ind_code):
            # if '2612' in fut_code:
            #     continue
            df_fut, df_ind = cal_ind_fut_data(date, fut_code, ind_code)
            pre_basis = (df_fut.loc[0, 'prev_settlement'] - df_ind.loc[0, 'prev_close']).round(2)
            basis = (df_fut.loc[0, 'settlement'] - df_ind.loc[0, 'close']).round(2)
            basis_pct = (basis - pre_basis) / df_ind.loc[0, 'prev_close']
            if basis_pct >= 0:
                res = f'收敛了：{format(basis_pct, ".2%")}'
            else:
                res = f'拉开了：{format(basis_pct * -1, ".2%")}'

            print(f'{fut_code}基差变化(昨→今)：{pre_basis}→{basis}, {res}')



exam_date = dt.datetime.now().strftime('%Y%m%d')
# exam_date = '20260515'

benchmark = '000300.XSHG'
# hengsheng_strategy_paths = rf'E:\code\generate_split_system\data\target\{exam_date}\{exam_date}_ABCD_000300.XSHG_zz1800_target.csv'
# hengsheng_strategy_paths = rf'E:\code\generate_split_system\data\target\{exam_date}\{exam_date}_TCHMD_000300.XSHG_zz1800_target.csv'
hengsheng_strategy_paths = rf'E:\code\generate_split_system\data\target\{exam_date}\{exam_date}_WBZD_000300.XSHG_zz1800_target.csv'
# hengsheng_strategy_paths = rf'E:\code\generate_split_system\data\target\{exam_date}\{exam_date}_WBZD_000300.XSHG_000906.XSHG_target.csv'

hengsheng_pos_paths = rf'E:\code\generate_split_system\data\standarddata\pos\{exam_date}\{exam_date}_haitongcredit_PBHSZX1H_pos.csv' # _haitongcredit_PBHSZX1H_pos, _haitong_PBZS1H_pos
analysis(exam_date, benchmark, hengsheng_strategy_paths, hengsheng_pos_paths)

print('--' * 10)
cal_basis(exam_date, 'IF')
print('--' * 10)
cal_basis(exam_date, 'IC')
print('--' * 10)
cal_basis(exam_date, 'IM')

-77242.20999999998
日期: 20260521
指数收益率为：-1.39%, 等权分配策略收益率：-1.30%, 策略收益率：-1.43%, 实际持仓收益率：-1.54%
等权分配策略超额：0.10%, 策略超额：-0.03%, 实际持仓超额：-0.15%
等权重为：0.45%, 实际持仓权重超过等权重的比例为: 15.45%
实际持仓中权重超过等权重部分的收益率为：-0.91%, 对应的等权重收益率为：-0.11%
实际持仓中权重小于等权重部分的收益率为：-0.63%, 对应的等权重收益率为：-1.18%
--------------------
IF2606基差变化(昨→今)：-45.1→-6.5, 收敛了：0.80%
IF2607基差变化(昨→今)：-82.3→-36.3, 收敛了：0.95%
IF2609基差变化(昨→今)：-121.3→-85.9, 收敛了：0.73%
IF2612基差变化(昨→今)：-172.5→-137.1, 收敛了：0.73%
--------------------
IC2606基差变化(昨→今)：-99.71→-15.24, 收敛了：0.98%
IC2607基差变化(昨→今)：-155.31→-89.04, 收敛了：0.77%
IC2609基差变化(昨→今)：-262.71→-174.24, 收敛了：1.02%
IC2612基差变化(昨→今)：-412.91→-329.44, 收敛了：0.96%
--------------------
IM2606基差变化(昨→今)：-126.51→-3.3, 收敛了：1.40%
IM2607基差变化(昨→今)：-199.91→-75.7, 收敛了：1.41%
IM2609基差变化(昨→今)：-347.91→-216.1, 收敛了：1.50%
IM2612基差变化(昨→今)：-557.51→-414.7, 收敛了：1.62%
